# Phase 6 — KGE Training & Evaluation on Google Colab

## Instructions
1. Make sure GPU is enabled: **Runtime → Change runtime type → T4 GPU**
2. Run each cell in order
3. When prompted, upload your 3 files: `train.txt`, `valid.txt`, `test.txt` (from your `kge/` folder)
4. At the end, download `kge_results.zip` and extract it into your local `kge/results/` folder

## Cell 1 — Install dependencies

In [ ]:
# Install PyKEEN with a compatible torch version
!pip install pykeen==1.10.2 -q
print('✅ PyKEEN installed')

✅ PyKEEN installed


In [ ]:

import pykeen
import torch
from importlib.metadata import version

# On récupère la version enregistrée par pip au lieu de l'attribut du module
pk_ver = version('pykeen')
torch_ver = torch.__version__

print(f'✅ PyKEEN version : {pk_ver}')
print(f'✅ PyTorch version: {torch_ver}')
print(f'✅ GPU available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'✅ GPU name       : {torch.cuda.get_device_name(0)}')

✅ PyKEEN version : 1.11.1
✅ PyTorch version: 2.10.0+cu128
✅ GPU available  : True
✅ GPU name       : Tesla T4


## Cell 2 — Upload your split files

In [ ]:
from google.colab import files
import os

os.makedirs('/content/kge', exist_ok=True)
print('Please upload your 3 files: train.txt, valid.txt, test.txt')
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = f'/content/kge/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  ✅ Saved {fname} → {dest}')

# Verify
for fname in ['train.txt', 'valid.txt', 'test.txt']:
    path = f'/content/kge/{fname}'
    if os.path.exists(path):
        lines = open(path).readlines()
        print(f'  {fname}: {len(lines):,} triplets')
    else:
        print(f'  ❌ Missing: {fname}')

Please upload your 3 files: train.txt, valid.txt, test.txt


Saving test.txt to test (1).txt
Saving train.txt to train (1).txt
Saving valid.txt to valid (1).txt
  ✅ Saved test (1).txt → /content/kge/test (1).txt
  ✅ Saved train (1).txt → /content/kge/train (1).txt
  ✅ Saved valid (1).txt → /content/kge/valid (1).txt
  train.txt: 22,790 triplets
  valid.txt: 2,848 triplets
  test.txt: 2,850 triplets


## Cell 3 — Load splits with TriplesFactory

In [ ]:
from pykeen.triples import TriplesFactory

TRAIN = '/content/kge/train.txt'
VALID = '/content/kge/valid.txt'
TEST  = '/content/kge/test.txt'

print('Loading train...')
tf_train = TriplesFactory.from_path(TRAIN, create_inverse_triples=False)
print(f'  → {tf_train.num_triples:,} triples | {tf_train.num_entities:,} entities | {tf_train.num_relations:,} relations')

print('Loading valid...')
tf_valid = TriplesFactory.from_path(
    VALID,
    entity_to_id=tf_train.entity_to_id,
    relation_to_id=tf_train.relation_to_id
)

print('Loading test...')
tf_test = TriplesFactory.from_path(
    TEST,
    entity_to_id=tf_train.entity_to_id,
    relation_to_id=tf_train.relation_to_id
)
print('✅ Splits loaded')

Loading train...
  → 22,790 triples | 14,784 entities | 369 relations
Loading valid...
Loading test...
✅ Splits loaded


## Cell 4 — Training configuration

In [ ]:
# ── Configuration ──────────────────────────────────────────
CONFIG = dict(
    embedding_dim    = 200,
    learning_rate    = 0.001,
    batch_size       = 512,
    num_epochs       = 100,   # ← Full 100 epochs, feasible on GPU
    negative_sampler = 'basic',
    random_seed      = 42,
)

MODELS = ['TransE', 'DistMult']   # Add 'RotatE', 'ComplEx' for bonus

print('Configuration:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')
print(f'Models to train: {MODELS}')

Configuration:
  embedding_dim: 200
  learning_rate: 0.001
  batch_size: 512
  num_epochs: 100
  negative_sampler: basic
  random_seed: 42
Models to train: ['TransE', 'DistMult']


## Cell 5 — Train all models (with GPU)

In [ ]:
# On installe la dernière version de PyKEEN et on met à jour les dépendances problématiques
%pip install --upgrade pykeen class-resolver typing-extensions

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 28.2 MB/s eta 0:00:00
  Attempting uninstall: pykeen
    Found existing installation: pykeen 1.10.2
    Uninstalling pykeen-1.10.2:
      Successfully uninstalled pykeen-1.10.2


In [ ]:
import json, time, os
from pykeen.pipeline import pipeline

RESULTS_DIR = '/content/kge/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

all_metrics = []

for model_name in MODELS:
    print(f'\n{'='*60}')
    print(f'Training {model_name}...')
    print(f'{'='*60}')

    out_dir = f'{RESULTS_DIR}/{model_name}'
    os.makedirs(out_dir, exist_ok=True)

    t0 = time.time()
    result = pipeline(
        training         = tf_train,
        validation       = tf_valid,
        testing          = tf_test,
        model            = model_name,
        model_kwargs     = dict(embedding_dim=CONFIG['embedding_dim']),
        training_kwargs  = dict(
            num_epochs = CONFIG['num_epochs'],
            batch_size = CONFIG['batch_size'],
        ),
        optimizer_kwargs = dict(lr=CONFIG['learning_rate']),
        negative_sampler = CONFIG['negative_sampler'],
        random_seed      = CONFIG['random_seed'],
        evaluator_kwargs = dict(filtered=True),
        device           = 'cuda' if __import__('torch').cuda.is_available() else 'cpu',
    )
    duration = time.time() - t0

    # Save model + results
    result.save_to_directory(out_dir)

    # Extract metrics
    metrics = {
        'model'          : model_name,
        'embedding_dim'  : CONFIG['embedding_dim'],
        'num_epochs'     : CONFIG['num_epochs'],
        'learning_rate'  : CONFIG['learning_rate'],
        'batch_size'     : CONFIG['batch_size'],
        'duration_s'     : round(duration),
    }
    try:
        mrr  = result.metric_results.get_metric('both.realistic.inverse_harmonic_mean_rank')
        h1   = result.metric_results.get_metric('both.realistic.hits_at_1')
        h3   = result.metric_results.get_metric('both.realistic.hits_at_3')
        h10  = result.metric_results.get_metric('both.realistic.hits_at_10')
        metrics.update({
            'MRR_filtered'    : round(float(mrr), 4),
            'Hits@1_filtered' : round(float(h1),  4),
            'Hits@3_filtered' : round(float(h3),  4),
            'Hits@10_filtered': round(float(h10), 4),
        })
    except Exception as e:
        print(f'  ⚠️ Could not extract metrics: {e}')
        # Fallback: try dict form
        try:
            d = result.metric_results.to_dict()
            for key, val in d.items():
                k = key.lower()
                if 'mrr' in k or 'inverse_harmonic' in k: metrics['MRR_filtered'] = round(float(val), 4)
                elif 'hits_at_1' in k: metrics['Hits@1_filtered'] = round(float(val), 4)
                elif 'hits_at_3' in k: metrics['Hits@3_filtered'] = round(float(val), 4)
                elif 'hits_at_10' in k: metrics['Hits@10_filtered'] = round(float(val), 4)
        except Exception:
            pass

    all_metrics.append(metrics)

    print(f'\n  ✅ {model_name} done in {duration:.0f}s')
    print(f'  MRR     : {metrics.get("MRR_filtered", "N/A")}')
    print(f'  Hits@1  : {metrics.get("Hits@1_filtered", "N/A")}')
    print(f'  Hits@3  : {metrics.get("Hits@3_filtered", "N/A")}')
    print(f'  Hits@10 : {metrics.get("Hits@10_filtered", "N/A")}')

print(f'\n✅ All models trained!')

INFO:pykeen.pipeline.api:Using device: cuda
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()



Training TransE...


Training epochs on cuda:0:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cuda:0:   0%|          | 0.00/45.0 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/45.0 [00:00<?, ?batch/s]

...

Training batches on cuda:0:   0%|          | 0.00/45.0 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/45.0 [00:00<?, ?batch/s]

Evaluating on cuda:0:   0%|          | 0.00/2.85k [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 1.23s seconds
INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=14784, num_relations=369, create_inverse_triples=False, num_triples=22790, path="/content/kge/train.txt") to file:///content/kge/results/DistMult/training_triples
INFO:pykeen.pipeline.api:Saved to directory: /content/kge/results/DistMult



  ✅ DistMult done in 51s
  MRR     : 0.2411
  Hits@1  : 0.1611
  Hits@3  : 0.2704
  Hits@10 : 0.3935

✅ All models trained!


## Cell 6 — Save comparison JSON + print summary table

In [ ]:
import json

compare_path = '/content/kge/comparaison_modeles.json'
with open(compare_path, 'w', encoding='utf-8') as f:
    json.dump({'configuration': CONFIG, 'resultats': all_metrics}, f, indent=2, ensure_ascii=False)
print(f'✅ Saved: {compare_path}')

# Print comparison table
print()
print('=' * 75)
print('COMPARISON TABLE (filtered metrics)')
print('=' * 75)
print(f'{"Model":<12} {"MRR":>8} {"Hits@1":>8} {"Hits@3":>8} {"Hits@10":>8} {"Time":>7}')
print('-' * 75)
for m in all_metrics:
    print(
        f'{m["model"]:<12}'
        f' {m.get("MRR_filtered","—"):>8}'
        f' {m.get("Hits@1_filtered","—"):>8}'
        f' {m.get("Hits@3_filtered","—"):>8}'
        f' {m.get("Hits@10_filtered","—"):>8}'
        f' {str(m.get("duration_s","—"))+"s":>7}'
    )
print('=' * 75)

✅ Saved: /content/kge/comparaison_modeles.json

COMPARISON TABLE (filtered metrics)
Model             MRR   Hits@1   Hits@3  Hits@10    Time
---------------------------------------------------------------------------
TransE         0.0937   0.0061   0.1333   0.2477     46s
DistMult       0.2411   0.1611   0.2704   0.3935     51s


## Cell 7 — Export embeddings for t-SNE / nearest-neighbor analysis

In [ ]:
# The result object from the LAST trained model is still available.
# This cell exports entity embeddings that your local t-SNE script can use.
# (Re-run the training loop if you need embeddings from a specific model)

import torch, json

for model_name in MODELS:
    model_dir = f'{RESULTS_DIR}/{model_name}'
    trained_model_path = f'{model_dir}/trained_model.pkl'

    if not __import__('os').path.exists(trained_model_path):
        print(f'  ⚠️ {model_name}: trained_model.pkl not found, skipping')
        continue

    trained_model = torch.load(trained_model_path, map_location='cpu', weights_only=False)

    try:
        # PyKEEN models store entity embeddings in .entity_representations[0]
        emb = trained_model.entity_representations[0](indices=None)
        emb_np = emb.detach().cpu().numpy()

        import numpy as np
        np.save(f'{model_dir}/entity_embeddings.npy', emb_np)
        print(f'  ✅ {model_name}: entity embeddings saved → shape {emb_np.shape}')
    except Exception as e:
        print(f'  ⚠️ {model_name}: could not extract embeddings: {e}')

# Also save entity-to-id mapping (needed for t-SNE labels)
entity_to_id = dict(tf_train.entity_to_id)
with open(f'{RESULTS_DIR}/entity_to_id.json', 'w') as f:
    json.dump(entity_to_id, f, indent=2)
print(f'  ✅ entity_to_id.json saved ({len(entity_to_id)} entities)')

  ✅ TransE: entity embeddings saved → shape (14784, 200)
  ✅ DistMult: entity embeddings saved → shape (14784, 200)
  ✅ entity_to_id.json saved (14784 entities)


## Cell 8 — Download everything as a ZIP

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/kge_results'
shutil.make_archive(zip_path, 'zip', '/content/kge/results')
print(f'✅ Archive created: {zip_path}.zip')

# Also download the comparison JSON separately
shutil.copy('/content/kge/comparaison_modeles.json', '/content/comparaison_modeles.json')

print('Downloading kge_results.zip...')
files.download('/content/kge_results.zip')

print('Downloading comparaison_modeles.json...')
files.download('/content/comparaison_modeles.json')

print('\n✅ Download complete!')
print('Next steps on your local machine:')
print('  1. Extract kge_results.zip → place contents in kge/results/')
print('  2. Copy comparaison_modeles.json → kge/comparaison_modeles.json')
print('  3. Run: python kge/analyse/tsne_visualization.py')
print('  4. Run: python kge/analyse/nearest_neighbors.py')

✅ Archive created: /content/kge_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download complete!
Next steps on your local machine:
  1. Extract kge_results.zip → place contents in kge/results/
  2. Copy comparaison_modeles.json → kge/comparaison_modeles.json
  3. Run: python kge/analyse/tsne_visualization.py
  4. Run: python kge/analyse/nearest_neighbors.py
